[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/NSA-hyperspectral-course-2026/blob/main/exercises/ex_03_ghg_plumes.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 💨 Exercises: Air quality & GHG plumes
### Space Academy 2026 · Hyperspectral course · Week 2 — Project scoping and use case definition (Practical)

Stress-test the simple methane band ratio (MBSI) from notebook 03 over the Kusmunda coal mine in India — one of Asia's largest open-cast mines — and find out what it really measures. This is a lesson in scientific scepticism: always ask what else could cause the signal.

These exercises accompany the theme notebook [`03_air_quality_ghg_plumes.ipynb`](https://github.com/Tnecniv-Teikram/813-hyperspectral-hackathon/blob/main/03_air_quality_ghg_plumes.ipynb). Run that notebook first
(or at least read it) — here we reuse its scene and indices, loaded with the course helper module `hsi_utils`.

> Work through the exercises in order — they get harder (⭐ → ⭐⭐⭐). Each has a hint; worked solutions are in `exercises/solutions/`.

**What you will practise**
- Locating index 'hot spots' and judging whether they are plausible
- Testing the sensitivity of an index to arbitrary choices (reference bands)
- Diagnosing surface-driven false positives with land-cover labels and spectra
- Using the GHG-plumes STAC metadata and provider plume products as reference data
- What a defensible gas-detection workflow needs

**Data**
- `20250219_053251_31_4001` — Kusmunda / Korba coalfield, Chhattisgarh, India (19 Feb 2025)
- `data/labels/…_worldcover2021_30m.tif` — ESA WorldCover 2021 on the Tanager grid
- Exercise 5 needs internet (Planet STAC catalogue).

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/NSA-hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## Setup

In [ ]:
# hsi_utils.py lives in the course folder (.. from here)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# On Google Colab: upload hsi_utils.py next to this notebook, change the line above to
# sys.path.insert(0, ".") and uncomment:
# %pip install -q h5py numpy scipy matplotlib scikit-learn pandas rasterio requests

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

Load the scene and the bands used in the theme notebook. `hu.get_scene()` finds the local HDF5 or downloads it (~1 GB) from Planet's STAC catalogue.

In [ ]:
SCENE_ID = "20250219_053251_31_4001"   # Kusmunda coal mine, Korba, Chhattisgarh, India (GHG-plumes)
scene = hu.get_scene(SCENE_ID)
print(scene.summary())

valid = scene.valid_mask()
R = {w: scene.read_band(w) for w in (2150, 2200, 2250, 2305, 2345, 2380, 2420, 480, 560, 665, 860, 1650)}

def mbsi(ref1=2250, ref2=2380, abs1=2305, abs2=2345):
    """Methane Band Suppression Index of notebook 03: shoulders / absorption − 1."""
    r_abs = (R[abs1] + R[abs2]) / 2
    return (R[ref1] + R[ref2]) / (2 * r_abs + 1e-6) - 1

MBSI = mbsi()
NDVI = hu.nd(R[860], R[665])
ALBEDO = np.mean([R[w] for w in (480, 560, 665, 860, 1650)], axis=0)
wc, purity = hu.load_labels(SCENE_ID)
ok = valid & np.isfinite(MBSI)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
axes[0].imshow(scene.rgb()); axes[0].set_title("True colour — open-cast coal mines in the south")
im = axes[1].imshow(np.where(ok, MBSI, np.nan), cmap="seismic", vmin=-0.1, vmax=0.1)
axes[1].set_title("MBSI (notebook 03)"); plt.colorbar(im, ax=axes[1], fraction=0.04)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Exercise 1 — Where are the 'methane hot spots'?  ⭐

Notebook 03 treats high MBSI as a possible methane signal.

1. Find the pixels in the top 0.5% of MBSI (valid pixels only) and overlay them in a bright colour on the
   true-colour image.
2. How many are there and what do these places look like? Are they on the mines, the power plants or somewhere
   else?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`t = np.nanpercentile(MBSI[ok], 99.5)`, `hot = ok & (MBSI > t)`. Overlay with
`ax.imshow(np.where(hot, 1, np.nan), cmap="autumn")` on top of `ax.imshow(scene.rgb())`.

</details>

## Exercise 2 — Sensitivity to the reference bands  ⭐⭐

MBSI compares the absorption bands (2305 and 2345 nm) with two "shoulder" reference bands (2250 and 2380 nm).
The choice of shoulders is arbitrary. Recompute MBSI with the shoulder pairs **(2200, 2420)** and
**(2150, 2420)** and compare:

1. the Pearson correlation of each version with the original, over valid pixels;
2. the overlap of the top-0.5% hot-spot sets, measured by the Jaccard index |A ∩ B| / |A ∪ B|.

Would a robust detection change this much?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`mbsi(ref1=2200, ref2=2420)` recomputes the index. Correlation: `np.corrcoef(a[ok], b[ok])[0, 1]`.

</details>

## Exercise 3 — Surface-driven false positives  ⭐⭐

Test the suspicion that MBSI measures the surface type:

1. Make box plots of MBSI per WorldCover class (tree cover 10, grassland 30, cropland 40, built-up 50,
   bare / mines 60, water 80), using pure pixels only.
2. Compute the correlation of MBSI with NDVI and with albedo (`ALBEDO`).
3. Which surface type would a naive MBSI threshold flag most often? Which one least?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`classes = [10, 30, 40, 50, 60, 80]`; for each class `m = ok & (wc == c) & (purity == 9)`. Class names are in
`hu.WORLDCOVER_CLASSES[c][0]`.

</details>

## Exercise 4 — Why do trees look like methane?  ⭐⭐⭐

Explain the false positives with spectra:

1. Plot the 2100–2450 nm part of the mean spectra of (a) the top-0.5% MBSI pixels, (b) pure bare/mine pixels
   (WorldCover 60) and (c) pure cropland, each **divided by its own value at 2250 nm**, so that only the shape
   remains. Mark the two shoulder and two absorption bands.
2. Compute MBSI on the course reference library spectra (`hu.load_library()`), which were measured in scenes
   where there is no reason to expect methane. Which materials give the highest MBSI?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Read a window with `scene.read_cube(...)` or loop over pixel positions with `scene.read_pixel`. For the library,
`wl, lib, meta = hu.load_library()` gives a dict of spectra; use `hu.band_index(wl, 2305)` etc. to pick bands.

</details>

## Exercise 5 — What does the GHG-plumes catalogue actually provide?  ⭐⭐

*Needs internet.* The STAC items in the `GHG-plumes` collection contain more than reflectance.

1. Fetch the STAC item of this scene (`hu.fetch_item(SCENE_ID, "GHG-plumes")`) and print its `notes`,
   `plume_provider_ids` and the list of asset keys.
2. The `ortho_ql_ch4` asset is a methane quick-look raster on the same grid as the reflectance, produced by the
   plume provider. Open it with `rasterio` and count the pixels flagged as methane.
3. Print the `notes` of all GHG-plumes items in the course catalogue index (`hu.load_catalog_index()`).
   Which scenes would you use to develop and test a methane detector?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`item["properties"]["notes"]`; `rasterio.open(item["assets"]["ortho_ql_ch4"]["href"]).read(1)` (nodata = 0).
Wrap the network calls in `try/except` so the notebook still runs offline.

</details>

## Exercise 6 — Reflection — from a band ratio to a defensible detection  ⭐

In a short markdown cell, describe the minimum workflow you would need before telling an environmental
regulator "there is a methane leak at this facility". Mention the input data level, the detection method, how
you would rule out surface false positives, and what independent evidence you would look for.

In [ ]:
# optional scratch space — answer in a markdown cell

<details><summary>💡 Hint</summary>

Think: radiance vs reflectance, matched filter with a CH₄ target signature, background statistics, plume shape
and wind direction, repeat observations, known infrastructure, comparison with provider products.

</details>

## Key takeaways

- A band ratio in the CH₄ window responds to surface SWIR shape (vegetation, minerals) as much as to gas.
- If a detection changes when you change arbitrary parameters, it is not a detection.
- Provider products and STAC notes are valuable reference data — use them to find true negatives, too.
- Real methane retrievals use radiance, a gas target spectrum and background statistics (matched filter).

---

> 📖 Data: Tanager — CC-BY-4.0 © Planet Labs PBC  
> 📖 Labels: ESA WorldCover 2021 v200 — CC-BY 4.0